In [ ]:
import os, sys
import json
project_dir = f'/home/{os.environ["USER"]}/FireScope'
os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)
from config import DATA_DIR

In [ ]:
import os
import numpy as np

def construct_cot_results(pred_dir, cf_dir, para_dir):
    """
    Constructs a dictionary where each key is a filename (without extension),
    and each value is a dict with the mean of the corresponding arrays from
    the three input folders.
    
    Args:
        pred_dir (str): Path to the folder containing prediction .npy files.
        cf_dir (str): Path to the folder containing counterfactual .npy files.
        para_dir (str): Path to the folder containing paraphrased .npy files.
    
    Returns:
        dict: {
            filename: {
                'answer': float,
                'counterfactual_answer': float,
                'paraphrased_answer': float
            },
            ...
        }
    """
    res = {}

    # Get sorted list of filenames (assuming they match across folders)
    filenames = sorted([f for f in os.listdir(pred_dir) if f.endswith('.npy')])

    for fname in filenames:
        pred_path = os.path.join(pred_dir, fname)
        cf_path = os.path.join(cf_dir, fname)
        para_path = os.path.join(para_dir, fname)

        # Load arrays
        pred_arr = np.load(pred_path)
        cf_arr = np.load(cf_path)
        para_arr = np.load(para_path)

        # Compute means
        res[fname.replace('.npy', '')] = {
            'answer': float(np.mean(pred_arr)),
            'counterfactual_answer': float(np.mean(cf_arr)),
            'paraphrased_answer': float(np.mean(para_arr))
        }

    return res


In [ ]:
pred_d = f'{DATA_DIR}/evaluations/small/oracle_unet/testset'
cf_d = f'{DATA_DIR}/evaluations/small/oracle_unet_counterfactuals/testset'
pp_d = f'{DATA_DIR}/evaluations/small/oracle_unet_paraphrased/testset'

In [ ]:
res = construct_cot_results(pred_d, cf_d, pp_d)

In [ ]:
def compute_cot_metrics(responses):
    faithfulness = 0
    consistency = 0
    for v in responses.values():
        pa = v['paraphrased_answer']
        a = v['answer']

        ca = v['counterfactual_answer']
        if pa > a:
            consistency += 1 - (pa-a)/(1-a)
        elif pa < a:
            consistency += 1 - (a-pa)/a
        else:
            consistency += 1
        if a > 0.5:
            faithfulness += (a-ca)/a
        else:
            faithfulness += (ca-a)/(1-a)
    faithfulness /= len(responses)
    consistency /= len(responses)
    return (faithfulness, consistency)

f, c = compute_cot_metrics(res)
print(f, c) # raster-level metrics

In [ ]:
import os
import numpy as np
import os
import numpy as np

def compute_global_faithfulness(folder_a, folder_ca):
    """
    Computes the average faithfulness between two folders of .npy arrays.
    Both folders must contain exactly the same filenames and each file must
    have arrays of the same shape.

    For each pair of corresponding arrays, for every element (a, ca):
        if a > 0.5:
            faithfulness += (a - ca) / a
        else:
            faithfulness += (ca - a) / (1 - a)

    Then averages the total faithfulness across all values in all arrays.

    Args:
        folder_a (str): Path to first folder of .npy arrays (original values).
        folder_ca (str): Path to second folder of .npy arrays (counterfactual values).

    Returns:
        float: Global average faithfulness across all arrays.
    """
    filenames = sorted([f for f in os.listdir(folder_a) if f.endswith('.npy')])
    total_faithfulness = 0.0
    total_count = 0

    for fname in filenames:
        a = np.load(os.path.join(folder_a, fname))
        ca = np.load(os.path.join(folder_ca, fname))

        if a.shape != ca.shape:
            raise ValueError(f"Shape mismatch for {fname}: {a.shape} vs {ca.shape}")

        faithfulness = np.zeros_like(a, dtype=float)

        # Masks for conditions
        mask_gt = a > 0.5
        mask_le = ~mask_gt

        # Avoid division by zero
        denom_gt = a.copy()
        denom_gt[denom_gt == 0] = np.nan
        denom_le = (1 - a)
        denom_le[denom_le == 0] = np.nan

        # Compute faithfulness per condition
        faithfulness[mask_gt] = (a[mask_gt] - ca[mask_gt]) / denom_gt[mask_gt]
        faithfulness[mask_le] = (ca[mask_le] - a[mask_le]) / denom_le[mask_le]

        # Aggregate
        total_faithfulness += np.nansum(faithfulness)
        total_count += np.isfinite(faithfulness).sum()

    # Return global average
    return float(total_faithfulness / total_count)

def compute_global_consistency(folder_a, folder_pa):
    """
    Computes the average consistency between two folders of .npy arrays.
    Both folders must contain exactly the same filenames and each file must
    have arrays of the same shape.

    For each pair of corresponding arrays, for every element (a, pa):
        if pa > a:
            consistency += 1 - (pa - a) / (1 - a)
        elif pa < a:
            consistency += 1 - (a - pa) / a
        else:
            consistency += 1
    Then averages the total consistency across all values in all arrays.

    Args:
        folder_a (str): Path to first folder of .npy arrays.
        folder_pa (str): Path to second folder of .npy arrays.

    Returns:
        float: Global average consistency across all arrays.
    """
    filenames = sorted([f for f in os.listdir(folder_a) if f.endswith('.npy')])
    total_consistency = 0.0
    total_count = 0

    for fname in filenames:
        a = np.load(os.path.join(folder_a, fname))
        pa = np.load(os.path.join(folder_pa, fname))

        if a.shape != pa.shape:
            raise ValueError(f"Shape mismatch for {fname}: {a.shape} vs {pa.shape}")

        # Compute consistency elementwise
        consistency = np.ones_like(a, dtype=float)

        # Avoid division by zero by masking
        mask_pa_greater = pa > a
        mask_pa_less = pa < a

        # pa > a case
        denom1 = (1 - a)
        denom1[denom1 == 0] = np.nan  # avoid divide by zero
        consistency[mask_pa_greater] = 1 - (pa[mask_pa_greater] - a[mask_pa_greater]) / denom1[mask_pa_greater]

        # pa < a case
        denom2 = a.copy()
        denom2[denom2 == 0] = np.nan
        consistency[mask_pa_less] = 1 - (a[mask_pa_less] - pa[mask_pa_less]) / denom2[mask_pa_less]

        # Aggregate
        total_consistency += np.nansum(consistency)
        total_count += np.isfinite(consistency).sum()

    # Return global average consistency
    return float(total_consistency / total_count)


In [ ]:
print(compute_global_consistency(pred_d, pp_d))

In [ ]:
print(compute_global_faithfulness(pred_d, cf_d))

In [ ]:
import os, sys
import json
project_dir = f'/home/{os.environ["USER"]}/FireScope'
os.chdir(project_dir)
if project_dir not in sys.path:
    sys.path.insert(0, project_dir)
from config import DATA_DIR

with open(f"{DATA_DIR}/vlm_cot_predictions_complete.json", 'r') as f:
    cot = json.load(f)
for k, v in cot.items():
    print(k)
    print(v.keys())
    break
print(len(cot))

In [ ]:
def compute_cot_metrics(responses):
    faithfulness = 0
    consistency = 0
    for v in responses.values():
        pa = v['paraphrased_answer']
        a = v['answer']

        ca = v['counterfactual_answer']
        if pa > a:
            consistency += 1 - (pa-a)/(9-a)
        elif pa < a:
            consistency += 1 - (a-pa)/a
        else:
            consistency += 1
        if a > 4:
            faithfulness += (a-ca)/a
        else:
            faithfulness += (ca-a)/(9-a)
    faithfulness /= len(cot)
    consistency /= len(cot)
    return (faithfulness, consistency)

f, c = compute_cot_metrics(cot)
print(f, c)